### **Modelos tempranos de alineamiento visual-semántico, representaciones iniciales y antecedentes del aprendizaje multimodal**
 
Este cuaderno acompaña la segunda sesión del curso y tiene como objetivo construir una comprensión sólida de los primeros enfoques que buscaron relacionar texto e imagen en un **espacio semántico compartido**. La idea central no es estudiar todavía los modelos fundacionales multimodales, sino entender los **fundamentos representacionales** que hicieron posible su desarrollo posterior.


#### **0. Criterios de reproducibilidad**

- Este cuaderno genera un **dataset sintético local** de imágenes y captions, por lo que no depende de Kaggle, COCO ni descargas de internet.
- El pipeline usa dependencias presentes en el entorno base del curso: `numpy`, `pandas`, `scikit-learn`, `matplotlib`, `gensim`, `torch`, `torchvision`, `Pillow`, `transformers`, `nltk` y `spacy`.
- Si el entorno fue construido con dependencias opcionales, se habilitan celdas extra con `sentence-transformers`, `open_clip_torch` y `faiss-cpu`.
- El objetivo del cuaderno **no** es reproducir CLIP completo, sino **explicar técnicamente** sus antecedentes.

In [ ]:
import os
import math
import json
import random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import resnet18

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

from gensim.models import Word2Vec

SEED = 225
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Dispositivo:", DEVICE)

#### **1. Multimodalidad y tareas base**

La multimodalidad combina **más de una modalidad** de información dentro de un mismo problema de aprendizaje. En este cuaderno trabajaremos con el caso clásico **imagen + texto**.

### Tareas base
1. **Image-text retrieval:** dado un texto, recuperar la imagen relevante o dada una imagen, recuperar el texto relevante.  
2. **Captioning:** dada una imagen, producir una descripción textual.  
3. **Clasificación multimodal:** usar ambas modalidades para predecir una etiqueta.  
4. **VQA:** responder una pregunta textual condicionada a una imagen.  

Para mantener la reproducibilidad, generaremos un corpus sintético con objetos geométricos, colores y posiciones. Esto nos permite estudiar:
- atributos visuales concretos,
- relación entre tokens y regiones,
- grounding de palabras como *rojo*, *círculo*, *arriba*,
- limitaciones sobre palabras abstractas o contextuales.

In [ ]:
ROOT = Path("semana2_demo")
IMG_DIR = ROOT / "images"
ROOT.mkdir(exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)

colors = {
    "rojo": (220, 40, 40),
    "azul": (40, 90, 220),
    "verde": (40, 170, 70),
    "amarillo": (230, 200, 50),
}
shapes = ["circulo", "cuadrado", "triangulo"]
positions = {
    "arriba_izquierda": (24, 24),
    "arriba_derecha": (104, 24),
    "abajo_izquierda": (24, 104),
    "abajo_derecha": (104, 104),
}
backgrounds = {
    "claro": (245, 245, 245),
    "gris": (220, 220, 220),
}

def draw_shape(img_size=160, shape="circulo", color_name="rojo", pos_name="arriba_izquierda", bg_name="claro"):
    bg = backgrounds[bg_name]
    color = colors[color_name]
    img = Image.new("RGB", (img_size, img_size), bg)
    draw = ImageDraw.Draw(img)
    cx, cy = positions[pos_name]
    size = 28
    x1, y1 = cx - size, cy - size
    x2, y2 = cx + size, cy + size

    if shape == "circulo":
        draw.ellipse([x1, y1, x2, y2], fill=color, outline=(20, 20, 20))
    elif shape == "cuadrado":
        draw.rectangle([x1, y1, x2, y2], fill=color, outline=(20, 20, 20))
    elif shape == "triangulo":
        draw.polygon([(cx, y1), (x1, y2), (x2, y2)], fill=color, outline=(20, 20, 20))
    else:
        raise ValueError(shape)

    return img

def make_caption(shape, color, pos):
    pos_text = pos.replace("_", " ")
    return f"un {shape} {color} en {pos_text}"

rows = []
idx = 0
for bg_name in backgrounds:
    for color_name in colors:
        for shape_name in shapes:
            for pos_name in positions:
                img = draw_shape(shape=shape_name, color_name=color_name, pos_name=pos_name, bg_name=bg_name)
                fname = f"img_{idx:03d}.png"
                img.save(IMG_DIR / fname)
                rows.append({
                    "image_id": idx,
                    "filename": fname,
                    "caption_es": make_caption(shape_name, color_name, pos_name),
                    "shape": shape_name,
                    "color": color_name,
                    "position": pos_name,
                    "background": bg_name,
                })
                idx += 1

df = pd.DataFrame(rows)
df.to_csv(ROOT / "mini_pairs.csv", index=False)
df.head()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
sample = df.sample(8, random_state=SEED).reset_index(drop=True)

for ax, row in zip(axes.ravel(), sample.to_dict(orient="records")):
    img = Image.open(IMG_DIR / row["filename"])
    ax.imshow(img)
    ax.set_title(row["caption_es"], fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

#### **2. Representación textual inicial: one-hot, bag-of-words y word embeddings**

La progresión histórica básica es:
- **one-hot:** representación exacta pero extremadamente dispersa
- **bag-of-words (BoW):** cuenta de ocurrencias en vocabulario
- **word embeddings:** vectores densos que capturan similitud geométrica.

En modelos tempranos multimodales, la parte textual del problema arranca precisamente aquí.

In [ ]:
captions = df["caption_es"].tolist()

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(captions)
vocab = vectorizer.get_feature_names_out()

print("Número de captions:", len(captions))
print("Tamaño del vocabulario:", len(vocab))
print("Primeras palabras:", vocab[:15])
print("Shape BoW:", X_bow.shape)

# one-hot para una palabra del vocabulario
word_to_idx = {w: i for i, w in enumerate(vocab)}
token = "rojo"
one_hot = np.zeros(len(vocab), dtype=np.float32)
one_hot[word_to_idx[token]] = 1.0

print("Índice de 'rojo':", word_to_idx[token])
print("Suma del vector one-hot:", one_hot.sum())

In [ ]:
# Entrenamos un Word2Vec pequeño sobre el corpus sintético.
tokenized = [c.lower().split() for c in captions]

w2v = Word2Vec(
    sentences=tokenized,
    vector_size=32,
    window=3,
    min_count=1,
    workers=1,
    epochs=300,
    seed=SEED,
)

def sentence_embedding(sentence, model):
    toks = sentence.lower().split()
    vecs = [model.wv[t] for t in toks if t in model.wv]
    if not vecs:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0)

print("Embedding('rojo')[:8] =", np.round(w2v.wv["rojo"][:8], 3))
print("Embedding oración[:8] =", np.round(sentence_embedding(captions[0], w2v)[:8], 3))

In [ ]:
# Similaridad semántica simple entre tokens en el embedding entrenado
tokens_demo = ["rojo", "azul", "circulo", "cuadrado", "arriba", "abajo"]
sim_table = pd.DataFrame(index=tokens_demo, columns=tokens_demo, dtype=float)

for a in tokens_demo:
    for b in tokens_demo:
        sim_table.loc[a, b] = np.dot(w2v.wv[a], w2v.wv[b]) / (
            np.linalg.norm(w2v.wv[a]) * np.linalg.norm(w2v.wv[b])
        )

sim_table.round(3)

##### **Comentario conceptual**

Los embeddings entrenados en este corpus son **estáticos**:  la palabra `rojo` tiene siempre el mismo vector, sin importar el contexto.  
Esto es suficiente para un primer curso sobre alineamiento temprano, pero no resuelve:
- polisemia,
- desambiguación contextual,
- pragmática,
- razonamiento composicional profundo.

#### **3. Representación visual inicial: BOVW, CNN features y region features**

Vamos a cubrir tres enfoques:

1. **BOVW:** palabras visuales sobre patches densos.
2. **CNN features:** representaciones aprendidas.
3. **Region features:** vectores por subregiones de la imagen.

> En el entorno publicado del curso no se fuerza OpenCV ni detectores externos. Por eso usamos una variante reproducible con `PIL + numpy + sklearn`.

In [ ]:
def image_to_patches(img: Image.Image, patch_size=8, stride=8):
    arr = np.array(img.convert("L"), dtype=np.float32) / 255.0
    patches = []
    h, w = arr.shape
    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patch = arr[y:y+patch_size, x:x+patch_size].reshape(-1)
            patches.append(patch)
    return np.array(patches, dtype=np.float32)

# Recolectar patches para BOVW
all_patches = []
for fn in df["filename"]:
    img = Image.open(IMG_DIR / fn)
    all_patches.append(image_to_patches(img))

all_patches = np.vstack(all_patches)
print("Patches totales:", all_patches.shape)

kmeans = MiniBatchKMeans(n_clusters=24, random_state=SEED, batch_size=1024)
kmeans.fit(all_patches)

def bovw_histogram(img: Image.Image, kmeans_model, patch_size=8, stride=8):
    patches = image_to_patches(img, patch_size=patch_size, stride=stride)
    codes = kmeans_model.predict(patches)
    hist = np.bincount(codes, minlength=kmeans_model.n_clusters).astype(np.float32)
    hist /= max(hist.sum(), 1.0)
    return hist

example_img = Image.open(IMG_DIR / df.iloc[0]["filename"])
hist0 = bovw_histogram(example_img, kmeans)
print("Hist BOVW shape:", hist0.shape, "Suma:", hist0.sum())

In [ ]:
# Visualización simple de la representación BOVW
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(np.arange(len(hist0)), hist0)
ax.set_title("Histograma BOVW de una imagen")
ax.set_xlabel("Visual word")
ax.set_ylabel("Frecuencia normalizada")
plt.show()

##### **CNN features**

Entrenaremos una CNN pequeña sobre el dataset sintético para predecir la tripleta `(shape, color, position)`.  
No es un modelo SOTA, su rol es producir un **embedding visual aprendido** y mostrar el paso desde features manuales a representaciones neuronales.

In [ ]:
label_shape = {x: i for i, x in enumerate(sorted(df["shape"].unique()))}
label_color = {x: i for i, x in enumerate(sorted(df["color"].unique()))}
label_pos = {x: i for i, x in enumerate(sorted(df["position"].unique()))}

transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
])

class ShapeDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(IMG_DIR / row["filename"]).convert("RGB")
        x = transform(img)
        y_shape = label_shape[row["shape"]]
        y_color = label_color[row["color"]]
        y_pos = label_pos[row["position"]]
        return x, y_shape, y_color, y_pos

train_df = df.sample(frac=0.8, random_state=SEED)
val_df = df.drop(train_df.index)

train_loader = DataLoader(ShapeDataset(train_df), batch_size=32, shuffle=True)
val_loader = DataLoader(ShapeDataset(val_df), batch_size=32, shuffle=False)

class SmallCNN(nn.Module):
    def __init__(self, emb_dim=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 20 * 20, 128),
            nn.ReLU(),
            nn.Linear(128, emb_dim)
        )
        self.head_shape = nn.Linear(emb_dim, len(label_shape))
        self.head_color = nn.Linear(emb_dim, len(label_color))
        self.head_pos = nn.Linear(emb_dim, len(label_pos))

    def forward(self, x):
        h = self.features(x)
        z = self.proj(h)
        return z, self.head_shape(z), self.head_color(z), self.head_pos(z)

cnn = SmallCNN().to(DEVICE)
opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)

def run_epoch(loader, train=True):
    cnn.train(train)
    total = 0.0
    for x, y_shape, y_color, y_pos in loader:
        x = x.to(DEVICE)
        y_shape = y_shape.to(DEVICE)
        y_color = y_color.to(DEVICE)
        y_pos = y_pos.to(DEVICE)

        z, out_shape, out_color, out_pos = cnn(x)
        loss = (
            F.cross_entropy(out_shape, y_shape) +
            F.cross_entropy(out_color, y_color) +
            F.cross_entropy(out_pos, y_pos)
        )

        if train:
            opt.zero_grad()
            loss.backward()
            opt.step()

        total += loss.item() * x.size(0)
    return total / len(loader.dataset)

for epoch in range(8):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    print(f"Epoca {epoch+1:02d} | train={tr:.4f} | val={va:.4f}")

In [ ]:
@torch.no_grad()
def cnn_image_embedding(img: Image.Image):
    x = transform(img).unsqueeze(0).to(DEVICE)
    z, _, _, _ = cnn(x)
    z = F.normalize(z, dim=-1)
    return z.squeeze(0).cpu().numpy()

z_demo = cnn_image_embedding(example_img)
print("CNN feature dim:", z_demo.shape)

In [ ]:
def fixed_regions(img: Image.Image):
    w, h = img.size
    return {
        "global": img,
        "sup_izq": img.crop((0, 0, w//2, h//2)),
        "sup_der": img.crop((w//2, 0, w, h//2)),
        "inf_izq": img.crop((0, h//2, w//2, h)),
        "inf_der": img.crop((w//2, h//2, w, h)),
        "centro": img.crop((w//4, h//4, 3*w//4, 3*h//4)),
    }

img_demo = Image.open(IMG_DIR / df.iloc[10]["filename"])
regions = fixed_regions(img_demo)

fig, axes = plt.subplots(2, 3, figsize=(8, 5))
for ax, (name, reg) in zip(axes.ravel(), regions.items()):
    ax.imshow(reg)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

#### **4. Espacio compartido imagen-texto: proyección lineal y similitud**

Planteamos el problema clásico:

$$
t = E_{text}(x), \qquad v = E_{img}(I)
$$

$$
z_t = W_t t, \qquad z_v = W_v v
$$

$$
s(I, x) = \cos(z_v, z_t)
$$

donde:
- `t` es la representación textual,
- `v` es la representación visual,
- `W_t`, `W_v` proyectan ambas modalidades al mismo espacio,
- `s` mide cercanía semántica.

In [ ]:
# Construcción de embeddings de texto e imagen
text_embs = np.vstack([sentence_embedding(c, w2v) for c in df["caption_es"]]).astype(np.float32)
img_embs = np.vstack([cnn_image_embedding(Image.open(IMG_DIR / fn).convert("RGB")) for fn in df["filename"]]).astype(np.float32)

print("Text emb matrix:", text_embs.shape)
print("Image emb matrix:", img_embs.shape)

In [ ]:
class SharedSpace(nn.Module):
    def __init__(self, dim_img, dim_txt, dim_shared=32):
        super().__init__()
        self.img_proj = nn.Linear(dim_img, dim_shared)
        self.txt_proj = nn.Linear(dim_txt, dim_shared)

    def forward(self, img_vec, txt_vec):
        zi = F.normalize(self.img_proj(img_vec), dim=-1)
        zt = F.normalize(self.txt_proj(txt_vec), dim=-1)
        return zi, zt

def margin_ranking_loss(zi, zt, margin=0.2):
    # positivos: diagonal
    sim = zi @ zt.T
    pos = sim.diag()

    # negativos: desplazamiento circular
    neg_txt = torch.roll(zt, shifts=1, dims=0)
    neg_img = torch.roll(zi, shifts=1, dims=0)

    neg_sim_1 = torch.sum(zi * neg_txt, dim=-1)
    neg_sim_2 = torch.sum(zt * neg_img, dim=-1)

    loss1 = torch.clamp(margin - pos + neg_sim_1, min=0.0)
    loss2 = torch.clamp(margin - pos + neg_sim_2, min=0.0)
    return (loss1.mean() + loss2.mean()) / 2.0

shared = SharedSpace(dim_img=img_embs.shape[1], dim_txt=text_embs.shape[1], dim_shared=32).to(DEVICE)
optimizer = torch.optim.Adam(shared.parameters(), lr=1e-3)

img_tensor = torch.tensor(img_embs, dtype=torch.float32, device=DEVICE)
txt_tensor = torch.tensor(text_embs, dtype=torch.float32, device=DEVICE)

for epoch in range(250):
    shared.train()
    zi, zt = shared(img_tensor, txt_tensor)
    loss = margin_ranking_loss(zi, zt, margin=0.2)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoca {epoch+1:03d} | loss={loss.item():.4f}")

In [ ]:
@torch.no_grad()
def shared_similarity_matrix(model, img_tensor, txt_tensor):
    model.eval()
    zi, zt = model(img_tensor, txt_tensor)
    sim = zi @ zt.T
    return sim.cpu().numpy()

sim = shared_similarity_matrix(shared, img_tensor, txt_tensor)

def retrieval_at_1(sim_matrix):
    preds = sim_matrix.argmax(axis=1)
    gold = np.arange(len(preds))
    return (preds == gold).mean()

r1 = retrieval_at_1(sim)
print("Retrieval@1 (imagen -> texto):", round(float(r1), 4))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim[:20, :20], cmap="viridis")
ax.set_title("Matriz de similitud en el espacio compartido (primeros 20 pares)")
ax.set_xlabel("Texto")
ax.set_ylabel("Imagen")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

#### **5. Pérdida de ranking/margen con ejemplo numérico simple**

La pérdida clásica de alineamiento temprano puede escribirse como:

$$
\mathcal{L} = \max(0, \alpha - s(I, x^+) + s(I, x^-))
$$

La intuición es simple:
- el par correcto `(I, x⁺)` debe tener similitud alta.
- el par incorrecto `(I, x⁻)` debe quedar al menos `α` por debajo.

In [ ]:
alpha = 0.2
s_pos = 0.78
s_neg = 0.61
loss_simple = max(0.0, alpha - s_pos + s_neg)
print("Ejemplo numérico de loss =", round(loss_simple, 4))

In [ ]:
# Visualizamos cómo cambia la pérdida al mover el negativo
neg_values = np.linspace(-0.2, 1.0, 200)
loss_curve = np.maximum(0.0, alpha - s_pos + neg_values)

plt.figure(figsize=(6, 3))
plt.plot(neg_values, loss_curve)
plt.axvline(s_pos - alpha, linestyle="--")
plt.title("Pérdida de margen en función de la similitud negativa")
plt.xlabel("s(I, x-)")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()

#### **6. Multimodal distributional semantics: BOVW + vector textual + SVD**

La idea histórica es fusionar:
- un vector textual,
- un vector visual,
- y comprimir/fundir ambos espacios mediante SVD.

Aquí construiremos una versión pedagógica **a nivel de conceptos** (`rojo`, `circulo`, `arriba`, etc.).

In [ ]:
# Construimos histogramas BOVW por imagen
bovw_matrix = np.vstack([
    bovw_histogram(Image.open(IMG_DIR / fn), kmeans)
    for fn in df["filename"]
]).astype(np.float32)

# Vocabulario conceptual
concepts = []
for col in ["shape", "color", "position"]:
    concepts.extend(sorted(df[col].unique()))
concepts = list(dict.fromkeys(concepts))

def concept_text_vector(concept):
    # embedding textual del token o promedio por partes si tiene underscore
    pieces = concept.replace("_", " ").split()
    vecs = [w2v.wv[p] for p in pieces if p in w2v.wv]
    if not vecs:
        return np.zeros(w2v.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0)

def concept_visual_vector(concept):
    mask = (
        (df["shape"] == concept) |
        (df["color"] == concept) |
        (df["position"] == concept)
    ).values
    return bovw_matrix[mask].mean(axis=0)

X_text = np.vstack([concept_text_vector(c) for c in concepts]).astype(np.float32)
X_vis = np.vstack([concept_visual_vector(c) for c in concepts]).astype(np.float32)
X_concat = np.hstack([X_text, X_vis])

svd = TruncatedSVD(n_components=8, random_state=SEED)
X_fused = svd.fit_transform(X_concat)

concept_df = pd.DataFrame(X_fused[:, :2], columns=["d1", "d2"])
concept_df["concept"] = concepts
concept_df

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(concept_df["d1"], concept_df["d2"])
for _, row in concept_df.iterrows():
    plt.text(row["d1"] + 0.02, row["d2"] + 0.02, row["concept"], fontsize=9)
plt.title("Espacio fusionado texto+visión (SVD sobre vectores concatenados)")
plt.xlabel("d1")
plt.ylabel("d2")
plt.tight_layout()
plt.show()

##### **Lectura conceptual**

Este bloque aproxima el espíritu de **multimodal distributional semantics**:
- el texto aporta regularidad lingüística,
- la visión aporta información perceptual,
- la reducción lineal compacta ambas fuentes en un espacio común.

Es especialmente útil para **palabras concretas** y menos efectivo para conceptos abstractos.

#### **7. Primeras variantes neuronales: CNN + skip-gram + concatenación**

No reproduciremos un paper completo, pero sí una **aproximación**:
- texto: embedding estático de oración,
- imagen: CNN embedding,
- fusión por concatenación,
- tarea: decidir si un par imagen-texto es correcto o incorrecto.

Esto refleja una etapa intermedia entre:
- ingeniería de features,
- y modelos de alineamiento aprendidos de forma más estructurada.

In [ ]:
# Construimos pares positivos y negativos
pair_rows = []
for i, row in df.iterrows():
    pair_rows.append((i, i, 1))
    neg = (i + 7) % len(df)
    pair_rows.append((i, neg, 0))

pair_df = pd.DataFrame(pair_rows, columns=["img_idx", "txt_idx", "label"])
pair_df = pair_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

X_pair = []
y_pair = []
for _, r in pair_df.iterrows():
    vec = np.concatenate([img_embs[r["img_idx"]], text_embs[r["txt_idx"]]])
    X_pair.append(vec)
    y_pair.append(r["label"])

X_pair = torch.tensor(np.vstack(X_pair), dtype=torch.float32, device=DEVICE)
y_pair = torch.tensor(np.array(y_pair), dtype=torch.float32, device=DEVICE).unsqueeze(1)

class PairMatcher(nn.Module):
    def __init__(self, dim_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_in, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

matcher = PairMatcher(X_pair.shape[1]).to(DEVICE)
opt_match = torch.optim.Adam(matcher.parameters(), lr=1e-3)

for epoch in range(120):
    matcher.train()
    logits = matcher(X_pair)
    loss = F.binary_cross_entropy_with_logits(logits, y_pair)

    opt_match.zero_grad()
    loss.backward()
    opt_match.step()

    if (epoch + 1) % 30 == 0:
        with torch.no_grad():
            preds = (torch.sigmoid(logits) > 0.5).float()
            acc = (preds == y_pair).float().mean().item()
        print(f"Epoca {epoch+1:03d} | loss={loss.item():.4f} | acc={acc:.4f}")

#### **8. Alineamiento a nivel de oración y regiones**

Ahora pasamos de la pregunta global:
> "¿esta imagen corresponde a este caption?"

a una más fina:
> "¿qué partes de la imagen son más relevantes para cada palabra o para la oración completa?"

Usaremos:
- embedding de la oración,
- embeddings de regiones fijas,
- similitud oración-región y token-región.

In [ ]:
# Embeddings de regiones
def region_embeddings(img: Image.Image):
    regs = fixed_regions(img)
    out = {}
    for name, reg in regs.items():
        out[name] = cnn_image_embedding(reg)
    return out

sample_idx = 3
sample_row = df.iloc[sample_idx]
sample_img = Image.open(IMG_DIR / sample_row["filename"]).convert("RGB")
sample_regs = region_embeddings(sample_img)
sample_sentence = sample_row["caption_es"]
sample_sent_emb = sentence_embedding(sample_sentence, w2v)

with torch.no_grad():
    t = torch.tensor(sample_sent_emb, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    zt = F.normalize(shared.txt_proj(t), dim=-1).cpu().numpy()[0]

region_scores = {}
with torch.no_grad():
    for name, vec in sample_regs.items():
        v = torch.tensor(vec, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        zv = F.normalize(shared.img_proj(v), dim=-1).cpu().numpy()[0]
        region_scores[name] = float(np.dot(zt, zv))

region_scores

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(8, 5))
for ax, (name, reg) in zip(axes.ravel(), fixed_regions(sample_img).items()):
    ax.imshow(reg)
    score = region_scores[name]
    ax.set_title(f"{name}\nscore={score:.3f}")
    ax.axis("off")

plt.suptitle(sample_sentence)
plt.tight_layout()
plt.show()

In [ ]:
# token -> región
tokens = sample_sentence.split()
token_region = []

with torch.no_grad():
    for tok in tokens:
        if tok not in w2v.wv:
            continue
        tok_vec = torch.tensor(w2v.wv[tok], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        ztok = F.normalize(shared.txt_proj(tok_vec), dim=-1).cpu().numpy()[0]

        row_scores = {}
        for name, vec in sample_regs.items():
            zv = F.normalize(shared.img_proj(torch.tensor(vec, dtype=torch.float32, device=DEVICE).unsqueeze(0)), dim=-1).cpu().numpy()[0]
            row_scores[name] = float(np.dot(ztok, zv))
        token_region.append({"token": tok, **row_scores})

token_region_df = pd.DataFrame(token_region)
token_region_df

In [ ]:
plt.figure(figsize=(9, 3))
plt.imshow(token_region_df.drop(columns=["token"]).values, cmap="magma", aspect="auto")
plt.yticks(range(len(token_region_df)), token_region_df["token"])
plt.xticks(range(len(token_region_df.columns)-1), token_region_df.columns[1:], rotation=45)
plt.colorbar()
plt.title("Alineamiento token-región (aproximación simple)")
plt.tight_layout()
plt.show()

#### **9. Grounding y límites de los enfoques tempranos**

Los enfoques tempranos funcionan bien cuando:
- el concepto es concreto,
- la señal visual es clara,
- y la relación texto-imagen es directa.

Pero fallan o se debilitan cuando aparecen:
- abstracciones,
- contexto discursivo,
- composicionalidad compleja,
- negación,
- intención,
- ironía,
- o regiones no localizadas por el método.

In [ ]:
limitaciones = pd.DataFrame({
    "tipo": ["objeto", "atributo visible", "posición", "relación compleja", "abstracción", "pragmática"],
    "ejemplo": ["círculo", "rojo", "arriba", "a la izquierda de", "justicia", "ironía"],
    "grounding_visual": ["alto", "alto", "medio", "medio-bajo", "bajo", "muy bajo"],
    "comentario": [
        "fácil de asociar con forma",
        "fácil si el color es dominante",
        "depende de la partición regional",
        "requiere composición espacial",
        "sin soporte visual directo",
        "requiere contexto y mundo"
    ]
})
limitaciones

#### **10. Embeddings estáticos, embeddings contextuales, LLM y MLLM**

##### **Distinciones esenciales**

| Tipo | Unidad base | Usa contexto | Modalidades | Ejemplo |
|---|---|---:|---|---|
| Embedding estático | palabra | No | texto | word2vec, GloVe |
| Embedding contextual | token en contexto | Sí | texto | BERT |
| LLM | secuencia textual | Sí | texto | GPT, Llama |
| MLLM | texto + visión (+ otras) | Sí | multimodal | BLIP-2, Flamingo, LLaVA |

El hilo histórico es:
1. representar palabras e imágenes,
2. alinear espacios,
3. contextualizar lenguaje,
4. integrar razonamiento y múltiples modalidades.

In [ ]:
summary_table = pd.DataFrame([
    {"familia": "Embedding estático", "contexto": "No", "multi_modal": "No", "fortaleza": "similitud léxica local", "limitación": "sin desambiguación contextual"},
    {"familia": "Embedding contextual", "contexto": "Sí", "multi_modal": "No", "fortaleza": "representación dependiente del contexto", "limitación": "solo texto"},
    {"familia": "LLM", "contexto": "Sí", "multi_modal": "No", "fortaleza": "razonamiento y generación textual", "limitación": "sin percepción visual nativa"},
    {"familia": "MLLM", "contexto": "Sí", "multi_modal": "Sí", "fortaleza": "lenguaje + visión + instrucciones", "limitación": "coste, sesgo, grounding imperfecto"},
])
summary_table

#### **11. Extensiones opcionales**

Las siguientes celdas **no son necesarias** pero aprovechan el Docker del curso cuando se instala con:
- `sentence-transformers`
- `faiss-cpu`
- `open_clip_torch`


In [ ]:
# EXTENSIÓN OPCIONAL 1: sentence-transformers
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
    print("sentence-transformers disponible.")
except Exception as e:
    HAS_ST = False
    print("sentence-transformers NO disponible:", e)

In [ ]:
# EXTENSIÓN OPCIONAL 2: open_clip_torch
try:
    import open_clip
    HAS_OPENCLIP = True
    print("open_clip_torch disponible.")
except Exception as e:
    HAS_OPENCLIP = False
    print("open_clip_torch NO disponible:", e)

##### **Ejercicio opcional A**
Reemplazar el embedding estático de oración por `sentence-transformers` y comparar Retrieval@1.

##### **Ejercicio opcional B**
Si `open_clip_torch` está disponible y existe conectividad para pesos, comparar:
- espacio compartido temprano entrenado en este cuaderno,
- embedding moderno tipo CLIP/OpenCLIP.

##### **Ejercicio opcional C**
Indexar embeddings con `faiss-cpu` para búsqueda top-k.

#### **12. Preguntas de discusión**

1. ¿Qué gana y qué pierde un modelo cuando sustituye BOVW por CNN features?  
2. ¿Por qué la pérdida de margen es natural en retrieval?  
3. ¿Qué parte del problema resuelven los embeddings contextuales que los embeddings estáticos no resuelven?  
4. ¿Por qué el grounding de palabras abstractas sigue siendo problemático incluso en modelos modernos?  
5. ¿Qué diferencia conceptual existe entre *alinear modalidades* y *razonar multimodalmente*?

#### **11. Extensión avanzada: `sentence-transformers`**

Esta sección sustituye el embedding estático de oración por un embedding **contextual** moderno.  
La meta no es cambiar el eje histórico del cuaderno, sino mostrar cómo mejora la representación textual cuando pasamos de:
- promedio de word vectors estáticos,
- a embeddings de oración entrenados con objetivos semánticos más robustos.


In [ ]:
# Verificación de disponibilidad
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
    print("sentence-transformers disponible.")
except Exception as e:
    HAS_ST = False
    print("sentence-transformers NO disponible:", e)

In [ ]:
# Carga de modelo contextual de oraciones
st_model = None
st_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

if HAS_ST:
    try:
        st_model = SentenceTransformer(st_model_name)
        print("Modelo cargado:", st_model_name)
    except Exception as e:
        print("No se pudo cargar el modelo. Motivo:", e)
        st_model = None

In [ ]:
def retrieval_at_1_from_embeddings(img_Z: np.ndarray, txt_Z: np.ndarray) -> float:
    sim = img_Z @ txt_Z.T
    preds = sim.argmax(axis=1)
    gold = np.arange(len(preds))
    return float((preds == gold).mean())

st_results = {}

if st_model is not None:
    st_text_embs = st_model.encode(
        df["caption_es"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)

    # Proyección lineal al espacio visual ya aprendido
    class STSharedSpace(nn.Module):
        def __init__(self, dim_img, dim_txt, dim_shared=64):
            super().__init__()
            self.img_proj = nn.Linear(dim_img, dim_shared)
            self.txt_proj = nn.Linear(dim_txt, dim_shared)

        def forward(self, img_vec, txt_vec):
            zi = F.normalize(self.img_proj(img_vec), dim=-1)
            zt = F.normalize(self.txt_proj(txt_vec), dim=-1)
            return zi, zt

    st_shared = STSharedSpace(dim_img=img_embs.shape[1], dim_txt=st_text_embs.shape[1], dim_shared=64).to(DEVICE)
    opt_st = torch.optim.Adam(st_shared.parameters(), lr=1e-3)

    img_tensor_st = torch.tensor(img_embs, dtype=torch.float32, device=DEVICE)
    txt_tensor_st = torch.tensor(st_text_embs, dtype=torch.float32, device=DEVICE)

    for epoch in range(300):
        zi, zt = st_shared(img_tensor_st, txt_tensor_st)
        loss = margin_ranking_loss(zi, zt, margin=0.2)

        opt_st.zero_grad()
        loss.backward()
        opt_st.step()

    with torch.no_grad():
        zi, zt = st_shared(img_tensor_st, txt_tensor_st)
        img_Z_st = zi.cpu().numpy()
        txt_Z_st = zt.cpu().numpy()

    st_results["retrieval_at_1"] = retrieval_at_1_from_embeddings(img_Z_st, txt_Z_st)
    st_results["text_dim"] = int(st_text_embs.shape[1])
    print(st_results)
else:
    print("Se omite la comparación contextual porque el modelo no está disponible.")

In [ ]:
# Comparación simple entre texto estático y texto contextual
comparison_rows = []

# baseline estático
with torch.no_grad():
    zi_base, zt_base = shared(img_tensor, txt_tensor)
    img_Z_base = zi_base.cpu().numpy()
    txt_Z_base = zt_base.cpu().numpy()

comparison_rows.append({
    "modelo_textual": "Word2Vec promedio + proyección",
    "retrieval_at_1": retrieval_at_1_from_embeddings(img_Z_base, txt_Z_base),
})

if "retrieval_at_1" in st_results:
    comparison_rows.append({
        "modelo_textual": "Sentence-Transformers + proyección",
        "retrieval_at_1": st_results["retrieval_at_1"],
    })

pd.DataFrame(comparison_rows)

#### **Comentario**

La diferencia clave no es solo dimensional o de performance.  
Un embedding de `sentence-transformers`:
- capta relaciones semánticas de oración,
- es más estable ante variaciones léxicas,
- y se acerca más a la transición histórica hacia embeddings **contextuales**.

Aun así, aquí seguimos en un esquema de **alineamiento dual simple**, no en un MLLM.

#### **12. Extensión avanzada: `FAISS` para indexación y recuperación**

Ahora añadimos un índice de búsqueda aproximada sobre embeddings visuales.  
Esto convierte el problema de similitud en un flujo más cercano a sistemas reales de recuperación:

1. proyectar imágenes a un espacio compartido,  
2. indexarlas,  
3. proyectar una consulta textual,  
4. recuperar vecinos más cercanos.  

> Esta sección funciona con los embeddings del cuaderno, incluso si no está disponible `sentence-transformers`.

In [ ]:
try:
    import faiss
    HAS_FAISS = True
    print("faiss disponible:", faiss.__version__)
except Exception as e:
    HAS_FAISS = False
    print("faiss NO disponible:", e)

In [ ]:
def build_faiss_index(vectors: np.ndarray):
    vecs = vectors.astype(np.float32).copy()
    faiss.normalize_L2(vecs)
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    return index

def query_faiss(index, query_vec: np.ndarray, k=5):
    q = query_vec.astype(np.float32).reshape(1, -1).copy()
    faiss.normalize_L2(q)
    scores, ids = index.search(q, k)
    return scores[0], ids[0]

faiss_demo_df = None

if HAS_FAISS:
    # índice sobre embeddings visuales del espacio compartido base
    faiss_index = build_faiss_index(img_Z_base)

    query_text = "un circulo rojo en arriba izquierda"
    q_text = sentence_embedding(query_text, w2v).astype(np.float32)

    with torch.no_grad():
        q_text_t = torch.tensor(q_text, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        q_text_z = F.normalize(shared.txt_proj(q_text_t), dim=-1).cpu().numpy()[0]

    scores, ids = query_faiss(faiss_index, q_text_z, k=5)

    faiss_demo_df = pd.DataFrame({
        "rank": np.arange(1, len(ids)+1),
        "image_id": ids,
        "score": scores,
        "caption_real": [df.iloc[i]["caption_es"] for i in ids],
        "filename": [df.iloc[i]["filename"] for i in ids],
    })
    faiss_demo_df
else:
    print("FAISS no está disponible.")

In [ ]:
# Visualización de resultados de búsqueda con FAISS
if faiss_demo_df is not None:
    query_text = "un circulo rojo en arriba izquierda"
    fig, axes = plt.subplots(1, min(5, len(faiss_demo_df)), figsize=(14, 3))
    if len(faiss_demo_df) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, faiss_demo_df.iterrows()):
        img = Image.open(IMG_DIR / row["filename"])
        ax.imshow(img)
        ax.set_title(f"rank={row['rank']}\nscore={row['score']:.3f}\n{row['caption_real']}", fontsize=8)
        ax.axis("off")
    plt.suptitle(f"Consulta FAISS: {query_text}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Evaluación Retrieval@K con FAISS
if HAS_FAISS:
    def faiss_recall_at_k(index, txt_vectors, gold_ids, k=1):
        hits = 0
        for qvec, gid in zip(txt_vectors, gold_ids):
            scores, ids = query_faiss(index, qvec, k=k)
            if int(gid) in set(map(int, ids)):
                hits += 1
        return hits / len(gold_ids)

    query_vectors = txt_Z_base
    gold_ids = np.arange(len(df))

    metrics = []
    for k in [1, 3, 5]:
        metrics.append({
            "metrica": f"Recall@{k}",
            "valor": faiss_recall_at_k(faiss_index, query_vectors, gold_ids, k=k),
        })
    pd.DataFrame(metrics)

#### **Comentario**

FAISS no cambia la teoría del alineamiento, pero sí cambia la **operacionalización**:
- entrenamiento del espacio compartido,
- indexación,
- búsqueda,
- evaluación top-k.

Esta conexión es muy útil para enlazar semana 2 con recuperación cruzada y búsqueda multimodal en semanas posteriores.

#### **13. Extensión avanzada: `OpenCLIP`**

Finalmente, añadimos una comparación con un modelo moderno de alineamiento contrastivo ya preentrenado.  
Aquí el objetivo no es reemplazar el contenido histórico, sino **medir la distancia** entre:

- un pipeline temprano construido en clase,
- y un modelo entrenado a gran escala con objetivos contrastivos modernos.

> **Nota:** esta sección es opcional. Requiere `open_clip_torch` y, en muchos entornos, descarga de pesos si no existen en caché.

In [ ]:
try:
    import open_clip
    HAS_OPENCLIP = True
    print("open_clip disponible.")
except Exception as e:
    HAS_OPENCLIP = False
    print("open_clip NO disponible:", e)

In [ ]:
openclip_results = None
openclip_model = None
openclip_preprocess = None
openclip_tokenizer = None

if HAS_OPENCLIP:
    try:
        model_name = "ViT-B-32"
        pretrained_name = "laion2b_s34b_b79k"

        openclip_model, _, openclip_preprocess = open_clip.create_model_and_transforms(
            model_name,
            pretrained=pretrained_name,
            device=DEVICE,
        )
        openclip_tokenizer = open_clip.get_tokenizer(model_name)
        openclip_model.eval()

        print("OpenCLIP cargado:", model_name, pretrained_name)
    except Exception as e:
        print("No se pudo cargar OpenCLIP:", e)
        openclip_model = None

In [ ]:
if openclip_model is not None:
    @torch.no_grad()
    def openclip_image_embeddings(file_list):
        feats = []
        for fn in file_list:
            img = Image.open(IMG_DIR / fn).convert("RGB")
            x = openclip_preprocess(img).unsqueeze(0).to(DEVICE)
            z = openclip_model.encode_image(x)
            z = F.normalize(z, dim=-1)
            feats.append(z.squeeze(0).cpu().numpy())
        return np.vstack(feats).astype(np.float32)

    @torch.no_grad()
    def openclip_text_embeddings(text_list):
        toks = openclip_tokenizer(text_list).to(DEVICE)
        z = openclip_model.encode_text(toks)
        z = F.normalize(z, dim=-1)
        return z.cpu().numpy().astype(np.float32)

    oc_img = openclip_image_embeddings(df["filename"].tolist())
    oc_txt = openclip_text_embeddings(df["caption_es"].tolist())

    oc_r1 = retrieval_at_1_from_embeddings(oc_img, oc_txt)
    openclip_results = {"retrieval_at_1": oc_r1, "dim": int(oc_img.shape[1])}
    print(openclip_results)
else:
    print("OpenCLIP no disponible para evaluación.")

In [ ]:
# Comparación consolidada
rows = [{
    "modelo": "Espacio compartido temprano (Word2Vec + CNN + ranking loss)",
    "retrieval_at_1": retrieval_at_1_from_embeddings(img_Z_base, txt_Z_base),
}]

if "retrieval_at_1" in st_results:
    rows.append({
        "modelo": "Sentence-Transformers + CNN + ranking loss",
        "retrieval_at_1": st_results["retrieval_at_1"],
    })

if openclip_results is not None:
    rows.append({
        "modelo": "OpenCLIP preentrenado",
        "retrieval_at_1": openclip_results["retrieval_at_1"],
    })

comparison_df = pd.DataFrame(rows)
comparison_df

In [ ]:
# Búsqueda OpenCLIP + FAISS, si ambos están disponibles
if HAS_FAISS and openclip_results is not None:
    oc_index = build_faiss_index(oc_img)

    query_text_oc = "un triangulo azul en abajo derecha"
    q_oc = openclip_text_embeddings([query_text_oc])[0]
    scores, ids = query_faiss(oc_index, q_oc, k=5)

    oc_search_df = pd.DataFrame({
        "rank": np.arange(1, len(ids)+1),
        "image_id": ids,
        "score": scores,
        "caption_real": [df.iloc[i]["caption_es"] for i in ids],
        "filename": [df.iloc[i]["filename"] for i in ids],
    })
    oc_search_df

In [ ]:
if HAS_FAISS and openclip_results is not None:
    fig, axes = plt.subplots(1, min(5, len(oc_search_df)), figsize=(14, 3))
    if len(oc_search_df) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, oc_search_df.iterrows()):
        img = Image.open(IMG_DIR / row["filename"])
        ax.imshow(img)
        ax.set_title(f"rank={row['rank']}\nscore={row['score']:.3f}\n{row['caption_real']}", fontsize=8)
        ax.axis("off")
    plt.suptitle(f"Consulta OpenCLIP + FAISS: {query_text_oc}")
    plt.tight_layout()
    plt.show()

#### **Interpretación sugerida**

Esta comparación debe leerse con cuidado:

- Si `OpenCLIP` supera ampliamente al modelo temprano, eso **no invalida** el enfoque histórico, muestra el efecto de **escala, datos, arquitectura y preentrenamiento contrastivo**.
- Si el modelo temprano rinde razonablemente bien en el dataset sintético, eso muestra que los conceptos de la semana 2 siguen siendo estructuralmente correctos.
- `sentence-transformers` suele mejorar la parte textual incluso sin cambiar el lado visual, lo que ilustra la transición entre embeddings estáticos y representaciones contextuales.
- `FAISS` hace visible la frontera entre **modelo** y **sistema de recuperación**.

#### **14. Preguntas finales de ampliación**

1. ¿Qué parte de la mejora moderna proviene del objetivo contrastivo, y cuál de la calidad de los encoders?  
2. ¿En qué escenarios un espacio compartido lineal sigue siendo una línea base competitiva?  
3. ¿Qué diferencia existe entre "buscar por similitud" y "razonar multimodalmente"?  
4. ¿Cómo cambia el problema cuando pasamos de captions cortos a preguntas, instrucciones o diálogos?  
5. ¿Qué componentes de este cuaderno serían reutilizables para la semana de recuperación cruzada?

In [ ]:
## Tus respuestas